# Spatial error analysis

## Setup

In [ ]:
#Configure features and spatial dependencies.
RUN_INSTALLS = False
if RUN_INSTALLS:
    %pip -q install pandas numpy scikit-learn matplotlib seaborn geopandas contextily libpysal esda
from pathlib import Path
from datetime import datetime
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
try:
    import geopandas as gpd
except Exception:
    gpd = None
try:
    from libpysal.weights import KNN
    from esda.moran import Moran
except Exception:
    KNN = None
    Moran = None
RANDOM_SEED = 42
TEST_SIZE = 0.3
REFERENCE_NOTE = 'CROME treated as a reference crop-map product rather than direct ground truth'
CLASS_LABELS = [
    'Winter wheat',
    'Winter barley',
    'Spring barley',
    'Beet (sugar beet / fodder beet)',
    'Maize',
    'Oilseed rape',
    'Potatoes',
    'Pulses / field beans and peas',
]
SEASONAL_WINDOWS = {
    'winter_establishment': ('2021-10-20', '2022-02-28'),
    'spring_growth': ('2022-03-01', '2022-05-31'),
    'summer_peak': ('2022-06-01', '2022-08-31'),
    'late_season': ('2022-09-01', '2022-09-30'),
}
S2_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
S2_INDICES = ['NDVI', 'NDRE', 'LSWI', 'EVI']
S1_BASE_FEATURES = ['VV', 'VH', 'VV_minus_VH', 'VV_div_VH']
S2_FEATURE_BANDS = [f'{season}_{band}' for season in SEASONAL_WINDOWS for band in (S2_BANDS + S2_INDICES)]
S1_FEATURE_BANDS = [f'{season}_{band}' for season in SEASONAL_WINDOWS for band in S1_BASE_FEATURES]
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start] + list(start.parents):
        if (candidate / 'code').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not find dissertation project root containing code/ and data/.')
PROJECT_ROOT = find_project_root()
WORK_DIR = PROJECT_ROOT / 'code' / '3_spatial_error_analysis'
PREV_DIR = PROJECT_ROOT / 'code' / '2_sentinel1_sentinel2_model'
BASELINE_DIR = PROJECT_ROOT / 'code' / '1_sentinel_2_baseline'
S1S2_SAMPLE_CSV = PREV_DIR / 'data' / 'processed' / 's1s2_2022_east_anglia_crome_8class_features.csv'
SAMPLE_POINTS_GPKG = BASELINE_DIR / 'processed_data' / 'crome_2022_east_anglia_8class_points_baseline.gpkg'
SAMPLE_POINTS_LAYER = 'points_baseline'
OUTPUTS = WORK_DIR / 'outputs'
TABLES = OUTPUTS / 'tables'
FIGURES = OUTPUTS / 'figures'
for folder in [TABLES, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
print('Project root:', PROJECT_ROOT)
print('Work dir:', WORK_DIR)
print('Feature CSV:', S1S2_SAMPLE_CSV)
print('Geopandas available:', gpd is not None)

## Load features

In [ ]:
#Load features and infer county labels.
def infer_county(sample_uid):
    text = str(sample_uid).lower()
    if 'suffolk' in text:
        return 'Suffolk'
    if 'norfolk' in text:
        return 'Norfolk'
    if 'cambridgeshire' in text:
        return 'Cambridgeshire'
    return np.nan
def clean_model_frame(df, feature_cols):
    missing = [col for col in ['analysis_class', 'class_id', 'sample_uid'] if col not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    out = df.copy()
    out = out[out['analysis_class'].isin(CLASS_LABELS)].copy()
    for col in feature_cols:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out['county'] = out['sample_uid'].map(infer_county)
    out = out.dropna(subset=feature_cols + ['analysis_class', 'sample_uid'])
    return out
if not S1S2_SAMPLE_CSV.exists():
    raise FileNotFoundError(f'Missing S1+S2 feature table: {S1S2_SAMPLE_CSV}')
raw = pd.read_csv(S1S2_SAMPLE_CSV)
s2_cols = [col for col in S2_FEATURE_BANDS if col in raw.columns]
s1_cols = [col for col in S1_FEATURE_BANDS if col in raw.columns]
feature_cols = s2_cols + s1_cols
if len(s2_cols) == 0 or len(s1_cols) == 0:
    raise ValueError(f'Expected both S2 and S1 features. Found S2={len(s2_cols)}, S1={len(s1_cols)}')
model_df = clean_model_frame(raw, feature_cols).reset_index(drop=True)
summary = pd.DataFrame([
    {'item': 'rows_used', 'value': len(model_df)},
    {'item': 's2_feature_count', 'value': len(s2_cols)},
    {'item': 's1_feature_count', 'value': len(s1_cols)},
    {'item': 'total_feature_count', 'value': len(feature_cols)},
    {'item': 'county_values', 'value': ', '.join(sorted(model_df['county'].dropna().unique()))},
])
summary.to_csv(TABLES / 'spatial_error_input_summary.csv', index=False)
display(summary)
display(model_df['analysis_class'].value_counts().reindex(CLASS_LABELS))

## Matched predictions

In [ ]:
#Train matched S2 optical and S1+S2 models.
def fit_rf(train_df, test_df, features, model_name):
    rf = RandomForestClassifier(
        n_estimators=500,
        class_weight='balanced_subsample',
        max_features='sqrt',
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    rf.fit(train_df[features], train_df['analysis_class'])
    pred = rf.predict(test_df[features])
    proba = rf.predict_proba(test_df[features])
    classes = list(rf.classes_)
    max_proba = proba.max(axis=1)
    ref_proba = [proba[i, classes.index(label)] if label in classes else np.nan for i, label in enumerate(test_df['analysis_class'])]
    metrics = {
        'model': model_name,
        'test_rows': len(test_df),
        'overall_agreement': accuracy_score(test_df['analysis_class'], pred),
        'balanced_accuracy': balanced_accuracy_score(test_df['analysis_class'], pred),
        'macro_f1': f1_score(test_df['analysis_class'], pred, labels=CLASS_LABELS, average='macro'),
        'weighted_f1': f1_score(test_df['analysis_class'], pred, labels=CLASS_LABELS, average='weighted'),
        'reference_note': REFERENCE_NOTE,
    }
    return rf, pred, max_proba, ref_proba, metrics
train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=model_df['analysis_class'],
)
train_df = model_df.loc[train_idx].copy()
test_df = model_df.loc[test_idx].copy()
s2_model, s2_pred, s2_conf, s2_ref_conf, s2_metrics = fit_rf(train_df, test_df, s2_cols, 'S2-only')
s1s2_model, s1s2_pred, s1s2_conf, s1s2_ref_conf, s1s2_metrics = fit_rf(train_df, test_df, feature_cols, 'S1+S2')
predictions = test_df[['sample_uid', 'analysis_class', 'class_id', 'county']].copy()
predictions = predictions.rename(columns={'analysis_class': 'reference_class'})
predictions['pred_s2'] = s2_pred
predictions['pred_s1s2'] = s1s2_pred
predictions['correct_s2'] = predictions['pred_s2'].eq(predictions['reference_class'])
predictions['correct_s1s2'] = predictions['pred_s1s2'].eq(predictions['reference_class'])
predictions['confidence_s2'] = s2_conf
predictions['confidence_s1s2'] = s1s2_conf
predictions['reference_probability_s2'] = s2_ref_conf
predictions['reference_probability_s1s2'] = s1s2_ref_conf
predictions['s1_added_value_status'] = np.select(
    [
        (~predictions['correct_s2']) & (predictions['correct_s1s2']),
        (predictions['correct_s2']) & (~predictions['correct_s1s2']),
        (~predictions['correct_s2']) & (~predictions['correct_s1s2']),
        (predictions['correct_s2']) & (predictions['correct_s1s2']),
    ],
    ['fixed_by_s1', 'degraded_after_s1', 'persistent_disagreement', 'both_correct'],
    default='unclassified',
)
metrics_df = pd.DataFrame([s2_metrics, s1s2_metrics])
metrics_df.to_csv(TABLES / 'matched_test_overall_metrics.csv', index=False)
predictions.to_csv(TABLES / 'matched_test_predictions_spatial_error.csv', index=False)
display(metrics_df)
display(predictions['s1_added_value_status'].value_counts())

## Summaries

In [ ]:
#Summarise disagreement by class and county.
def error_rate_summary(df, group_cols):
    grouped = df.groupby(group_cols, dropna=False)
    out = grouped.agg(
        n=('sample_uid', 'count'),
        s2_error_rate=('correct_s2', lambda x: 1 - x.mean()),
        s1s2_error_rate=('correct_s1s2', lambda x: 1 - x.mean()),
        fixed_by_s1=('s1_added_value_status', lambda x: (x == 'fixed_by_s1').sum()),
        degraded_after_s1=('s1_added_value_status', lambda x: (x == 'degraded_after_s1').sum()),
        persistent_disagreement=('s1_added_value_status', lambda x: (x == 'persistent_disagreement').sum()),
    ).reset_index()
    out['delta_error_rate_s1s2_minus_s2'] = out['s1s2_error_rate'] - out['s2_error_rate']
    return out.sort_values('s1s2_error_rate', ascending=False)
class_error = error_rate_summary(predictions, ['reference_class'])
county_error = error_rate_summary(predictions, ['county'])
county_class_error = error_rate_summary(predictions, ['county', 'reference_class'])
class_error.to_csv(TABLES / 'error_rate_by_class.csv', index=False)
county_error.to_csv(TABLES / 'error_rate_by_county.csv', index=False)
county_class_error.to_csv(TABLES / 'error_rate_by_county_and_class.csv', index=False)
s1s2_errors = predictions[~predictions['correct_s1s2']].copy()
confusion_pairs = (
    s1s2_errors.groupby(['reference_class', 'pred_s1s2'])
    .size()
    .reset_index(name='n')
    .sort_values('n', ascending=False)
)
confusion_pairs.to_csv(TABLES / 'top_s1s2_confusion_pairs.csv', index=False)
display(class_error)
display(county_error)
display(confusion_pairs.head(15))

## Figures

In [ ]:
#Plot disagreement and confusion results.
plot_class = class_error.melt(
    id_vars='reference_class',
    value_vars=['s2_error_rate', 's1s2_error_rate'],
    var_name='model',
    value_name='error_rate',
)
plot_class['model'] = plot_class['model'].map({'s2_error_rate': 'S2-only', 's1s2_error_rate': 'S1+S2'})
plt.figure(figsize=(11, 5.5))
sns.barplot(data=plot_class, x='reference_class', y='error_rate', hue='model')
plt.xlabel('CROME reference class')
plt.ylabel('Disagreement rate')
plt.ylim(0, 1)
plt.xticks(rotation=35, ha='right')
plt.title('Class-level disagreement with CROME reference labels')
plt.tight_layout()
plt.savefig(FIGURES / 'class_level_disagreement_rates.png', dpi=300)
plt.show()
plot_county = county_error.melt(
    id_vars='county',
    value_vars=['s2_error_rate', 's1s2_error_rate'],
    var_name='model',
    value_name='error_rate',
)
plot_county['model'] = plot_county['model'].map({'s2_error_rate': 'S2-only', 's1s2_error_rate': 'S1+S2'})
plt.figure(figsize=(7.5, 4.5))
sns.barplot(data=plot_county, x='county', y='error_rate', hue='model')
plt.xlabel('Held sample county inferred from sample_uid')
plt.ylabel('Disagreement rate')
plt.ylim(0, 1)
plt.title('County-level disagreement with CROME reference labels')
plt.tight_layout()
plt.savefig(FIGURES / 'county_level_disagreement_rates.png', dpi=300)
plt.show()
cm = confusion_matrix(predictions['reference_class'], predictions['pred_s1s2'], labels=CLASS_LABELS)
cm_df = pd.DataFrame(cm, index=CLASS_LABELS, columns=CLASS_LABELS)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted class')
plt.ylabel('CROME reference class')
plt.title('S1+S2 confusion matrix for matched spatial-error test set')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES / 's1s2_matched_test_confusion_matrix.png', dpi=300)
plt.show()

## Spatial join

In [ ]:
#Join predictions to sample geometry.
spatial_predictions = None
geometry_note = []
if gpd is None:
    geometry_note.append('geopandas unavailable; geometry join skipped')
elif not SAMPLE_POINTS_GPKG.exists():
    geometry_note.append(f'sample point GeoPackage not found: {SAMPLE_POINTS_GPKG}')
else:
    try:
        points = gpd.read_file(SAMPLE_POINTS_GPKG, layer=SAMPLE_POINTS_LAYER)
    except Exception:
        points = gpd.read_file(SAMPLE_POINTS_GPKG)
    if 'sample_uid' not in points.columns:
        geometry_note.append('sample_uid missing from point GeoPackage; geometry join skipped')
    else:
        keep_cols = ['sample_uid', 'geometry']
        extra_cols = [col for col in ['analysis_class', 'class_id'] if col in points.columns]
        spatial_predictions = points[keep_cols + extra_cols].merge(predictions, on='sample_uid', how='inner')
        if len(spatial_predictions) == 0:
            geometry_note.append('geometry join produced zero matched rows')
        else:
            spatial_predictions.to_file(WORK_DIR / 'outputs' / 'matched_test_predictions_spatial_error.gpkg', layer='matched_test_predictions', driver='GPKG')
            geometry_note.append(f'geometry join successful: {len(spatial_predictions)} matched test samples')
            plot_gdf = spatial_predictions.to_crs('EPSG:27700') if spatial_predictions.crs else spatial_predictions
            fig, axes = plt.subplots(1, 2, figsize=(12, 6))
            plot_gdf.plot(ax=axes[0], column='correct_s2', categorical=True, markersize=9, legend=True)
            axes[0].set_title('S2-only agreement with CROME labels')
            axes[0].set_axis_off()
            plot_gdf.plot(ax=axes[1], column='s1_added_value_status', categorical=True, markersize=9, legend=True)
            axes[1].set_title('Effect of adding Sentinel-1 SAR')
            axes[1].set_axis_off()
            plt.tight_layout()
            plt.savefig(FIGURES / 'spatial_error_points_map.png', dpi=300)
            plt.show()
print('\n'.join(geometry_note))

## Spatial autocorrelation

In [ ]:
#Estimate spatial autocorrelation of errors.
moran_rows = []
if spatial_predictions is None or len(spatial_predictions) == 0:
    moran_rows.append({'model': 'S1+S2', 'status': 'skipped_no_geometry', 'moran_i': np.nan, 'p_sim': np.nan})
elif KNN is None or Moran is None:
    moran_rows.append({'model': 'S1+S2', 'status': 'skipped_missing_libpysal_esda', 'moran_i': np.nan, 'p_sim': np.nan})
else:
    g = spatial_predictions.to_crs('EPSG:27700') if spatial_predictions.crs else spatial_predictions
    coords = np.column_stack([g.geometry.x, g.geometry.y])
    k = min(8, len(g) - 1)
    if k < 2:
        moran_rows.append({'model': 'S1+S2', 'status': 'skipped_too_few_points', 'moran_i': np.nan, 'p_sim': np.nan})
    else:
        weights = KNN.from_array(coords, k=k)
        weights.transform = 'R'
        values = (~g['correct_s1s2']).astype(int).to_numpy()
        moran = Moran(values, weights, permutations=999)
        moran_rows.append({'model': 'S1+S2', 'status': 'computed', 'k_neighbors': k, 'moran_i': moran.I, 'p_sim': moran.p_sim})
moran_df = pd.DataFrame(moran_rows)
moran_df.to_csv(TABLES / 's1s2_error_moran_i.csv', index=False)
display(moran_df)

## Output check

In [ ]:
#Check outputs
expected_outputs = [
    TABLES / 'spatial_error_input_summary.csv',
    TABLES / 'matched_test_overall_metrics.csv',
    TABLES / 'matched_test_predictions_spatial_error.csv',
    TABLES / 'error_rate_by_class.csv',
    TABLES / 'error_rate_by_county.csv',
    TABLES / 'error_rate_by_county_and_class.csv',
    TABLES / 'top_s1s2_confusion_pairs.csv',
    TABLES / 's1s2_error_moran_i.csv',
    FIGURES / 'class_level_disagreement_rates.png',
    FIGURES / 'county_level_disagreement_rates.png',
    FIGURES / 's1s2_matched_test_confusion_matrix.png',
]
checklist = pd.DataFrame({'path': [str(p) for p in expected_outputs], 'exists': [p.exists() for p in expected_outputs]})
checklist.to_csv(TABLES / 'output_checklist.csv', index=False)
display(checklist)